# 🧹 Data Cleaning Notebook
**Semi-Automated Classification Workflow**

Supported use cases:
- Customer Churn Prediction
- Credit Risk Prediction
- Loan Default Prediction

> **Goal:** Fix data quality issues found during EDA and produce a clean dataset ready for modeling.

---

## Section 1 — Load Libraries

Load all required libraries for data cleaning.

In [ ]:
import pandas as pd
import numpy as np

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print('Libraries loaded successfully.')

---
## Section 2 — Load Dataset

Upload your raw CSV file (the same file used in the EDA notebook).

In [ ]:
from google.colab import files

# Upload CSV file
uploaded = files.upload()
filename = list(uploaded.keys())[0]

df = pd.read_csv(filename)

# Keep a copy of original shape for the final summary
original_shape = df.shape

print(f'File loaded: {filename}')
print(f'Dataset Shape: {df.shape[0]} rows x {df.shape[1]} columns')

---
## Section 3 — Drop Irrelevant Features

Remove columns that are not useful for modeling — for example, ID columns, email addresses, phone numbers, or any columns flagged during EDA.

> **Action required:** Add the column names you want to drop in the list below.

In [ ]:
# ✏️ List the columns you want to remove
# Leave the list empty [] if there is nothing to drop
cols_to_drop = ['CustomerID', 'Email', 'Phone']  # <-- Edit this

# -----------------------------------------------
# Keep only columns that actually exist
cols_to_drop = [col for col in cols_to_drop if col in df.columns]

if not cols_to_drop:
    print('No columns to drop. Skipping this step.')
else:
    df = df.drop(columns=cols_to_drop)
    print(f'Dropped {len(cols_to_drop)} column(s): {cols_to_drop}')
    print(f'Remaining Shape: {df.shape[0]} rows x {df.shape[1]} columns')

---
## Section 4 — Remove Duplicate Rows

Duplicate rows are exact copies of another row. They can distort training and must be removed.

In [ ]:
duplicates_before = df.duplicated().sum()

df = df.drop_duplicates()
df = df.reset_index(drop=True)

duplicates_after = df.duplicated().sum()
duplicates_removed = duplicates_before - duplicates_after

print(f'Duplicate Rows Before : {duplicates_before}')
print(f'Duplicate Rows After  : {duplicates_after}')
print(f'Total Removed         : {duplicates_removed}')

if duplicates_removed > 0:
    print('\n⚠️  Duplicates removed.')
else:
    print('\n✅ No duplicates found.')

---
## Section 5 — Handle Missing Values

Fill missing values using simple default rules:
- **Numerical columns** → filled with the **median**
- **Categorical columns** → filled with the **mode** (most frequent value)

In [ ]:
missing_summary = []

for col in df.columns:
    missing_count = df[col].isnull().sum()

    if missing_count == 0:
        continue

    if df[col].dtype in ['int64', 'float64']:
        fill_value = df[col].median()
        method = 'Median'
    else:
        fill_value = df[col].mode()[0]
        method = 'Mode'

    df[col] = df[col].fillna(fill_value)

    missing_summary.append({
        'Feature': col,
        'Missing Count': missing_count,
        'Method Used': method
    })

if not missing_summary:
    print('No missing values found. Nothing to fill.')
else:
    missing_summary_df = pd.DataFrame(missing_summary)
    print(f'Missing value features fixed: {len(missing_summary_df)}')
    display(missing_summary_df)

# Store for summary
n_missing_fixed = len(missing_summary)

---
## Section 6 — Fix Data Types

Some columns may have incorrect data types — for example, numbers stored as text (`object`). This step attempts to safely convert them to the correct numeric type.

In [ ]:
dtype_fixes = []

for col in df.columns:
    if df[col].dtype == 'object':
        # Try converting to numeric
        converted = pd.to_numeric(df[col], errors='coerce')

        # Only apply if most values converted successfully (less than 5% became NaN)
        new_nulls = converted.isnull().sum()
        original_nulls = df[col].isnull().sum()
        extra_nulls = new_nulls - original_nulls
        null_rate = extra_nulls / len(df)

        if null_rate < 0.05:
            old_dtype = str(df[col].dtype)
            df[col] = converted
            dtype_fixes.append({
                'Column': col,
                'Old Data Type': old_dtype,
                'New Data Type': str(df[col].dtype)
            })

if not dtype_fixes:
    print('No data type issues found. Nothing to fix.')
else:
    dtype_fix_df = pd.DataFrame(dtype_fixes)
    print(f'Data type fixes applied: {len(dtype_fix_df)}')
    display(dtype_fix_df)

# Store for summary
n_dtype_fixed = len(dtype_fixes)

---
## Section 7 — Handle Invalid Values

Some numerical columns may contain obviously invalid values, such as negative ages or negative income. This step flags and replaces them with the column median.

> The list of columns to check is configurable below.

In [ ]:
# ✏️ Define columns that must be non-negative (values below 0 are invalid)
# Add or remove column names based on your dataset
non_negative_cols = ['age', 'income', 'loan_amount', 'credit_score',
                     'tenure', 'balance', 'salary', 'amount']

# Keep only columns that actually exist in the dataset
cols_to_check = [col for col in non_negative_cols if col in df.columns]

invalid_summary = []

for col in cols_to_check:
    invalid_mask = df[col] < 0
    invalid_count = invalid_mask.sum()

    if invalid_count > 0:
        # Replace invalid values with median
        median_val = df.loc[~invalid_mask, col].median()
        df.loc[invalid_mask, col] = median_val

        invalid_summary.append({
            'Feature': col,
            'Invalid Records Found': invalid_count
        })

if not invalid_summary:
    print('No invalid values detected in the checked columns.')
    if not cols_to_check:
        print('Note: No matching columns found. Update non_negative_cols if needed.')
else:
    invalid_df = pd.DataFrame(invalid_summary)
    print(f'Columns with invalid values fixed: {len(invalid_df)}')
    display(invalid_df)

# Store for summary
n_invalid_fixed = len(invalid_summary)

---
## Section 8 — Handle Outliers

Extreme values can negatively affect model training. This step caps outliers using the **IQR method**:
- Values below `Q1 - 1.5×IQR` are capped at the lower bound.
- Values above `Q3 + 1.5×IQR` are capped at the upper bound.

Columns with fewer than 10 unique values are skipped (likely binary or ordinal flags).

In [ ]:
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()

outlier_summary = []

for col in numerical_cols:
    # Skip columns with very few unique values
    if df[col].nunique() < 10:
        continue

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers_before = ((df[col] < lower) | (df[col] > upper)).sum()

    # Cap the values
    df[col] = df[col].clip(lower=lower, upper=upper)

    outliers_after = ((df[col] < lower) | (df[col] > upper)).sum()

    if outliers_before > 0:
        outlier_summary.append({
            'Feature': col,
            'Outlier Count Before': outliers_before,
            'Outlier Count After': outliers_after
        })

if not outlier_summary:
    print('No outliers detected. Nothing to cap.')
else:
    outlier_df = pd.DataFrame(outlier_summary)
    print(f'Outlier features processed: {len(outlier_df)}')
    display(outlier_df)

# Store for summary
n_outlier_features = len(outlier_summary)

---
## Section 9 — Cleaning Validation

Verify that all cleaning steps worked correctly. The dataset should now have:
- Zero missing values
- Zero duplicate rows
- Correct data types

In [ ]:
val_missing = df.isnull().sum().sum()
val_duplicates = df.duplicated().sum()

print('=== Cleaning Validation ===')
print(f'Missing Values Remaining  : {val_missing}')
print(f'Duplicate Rows Remaining  : {val_duplicates}')
print(f'Final Dataset Shape       : {df.shape[0]} rows x {df.shape[1]} columns')

print('\n--- Data Types After Cleaning ---')
print(df.dtypes.to_string())

print('\n--- Validation Result ---')
if val_missing == 0 and val_duplicates == 0:
    print('✅ Dataset passed validation. Ready for modeling.')
else:
    if val_missing > 0:
        print(f'⚠️  {val_missing} missing values still remain.')
    if val_duplicates > 0:
        print(f'⚠️  {val_duplicates} duplicate rows still remain.')

---
## Section 10 — Save Clean Dataset

Save the cleaned dataset as `clean_dataset.csv` and download it to your local machine.

In [ ]:
output_filename = 'clean_dataset.csv'

df.to_csv(output_filename, index=False)

print(f'Clean dataset saved as: {output_filename}')
print(f'Shape: {df.shape[0]} rows x {df.shape[1]} columns')

# Download the file to your local machine
files.download(output_filename)

---
## Section 11 — Cleaning Summary

A complete summary of all cleaning steps applied to the dataset.

In [ ]:
print('=' * 52)
print('             CLEANING SUMMARY')
print('=' * 52)
print(f'  Original Dataset Shape       : {original_shape[0]} rows x {original_shape[1]} columns')
print(f'  Duplicate Rows Removed       : {duplicates_removed}')
print(f'  Missing Value Features Fixed : {n_missing_fixed}')
print(f'  Data Type Fixes Applied      : {n_dtype_fixed}')
print(f'  Invalid Values Fixed         : {n_invalid_fixed}')
print(f'  Outlier Features Processed   : {n_outlier_features}')
print(f'  Final Dataset Shape          : {df.shape[0]} rows x {df.shape[1]} columns')
print('=' * 52)
print()
print('  ✅ Next Step: Run Baseline Benchmark Notebook')
print('=' * 52)